# Notebook 04: TTI User Identity — AWS Calls Attributed to the Teleport User

## Overview

Notebooks 01–03 proved that Teleport identity flows through the gateway into every
tool call, and that Cedar policies control which tools a user can reach. But the
Lambda still runs under its IAM execution role — CloudTrail shows the role, not the user.

This notebook closes that gap using **IAM Identity Center Trusted Token Issuers (TTI)**.
The interceptor validates the caller through Identity Center, then assumes a user-scoped
IAM role with the user's email as the `RoleSessionName`. Every AWS API call made with
those credentials is audited as the user, not the agent.

```
Teleport id_token (Authorization header)
  → Interceptor: sso-oidc:CreateTokenWithIAM  → IC validates user identity (TTI)
  → Interceptor: sts:AssumeRole (session=user email) → user-scoped temp credentials
  → Tool Lambda: sts:GetCallerIdentity         → ARN contains user email
  → CloudTrail:  assumed-role/.../jeffrey.ellin@goteleport.com
```

## Prerequisites

- Notebooks 01, 02, and 03 completed (interceptor, AVP, and gateway are deployed)
- `.env` populated with AWS credentials, `TELEPORT_CLUSTER`, `RESOURCE_PREFIX`, and `DEMO_USER_EMAILS`

> **One manual step required:** Identity-Enhanced Sessions must be enabled in the
> IAM Identity Center console before running Phase 0.
> **IAM Identity Center → Settings → Enable identity-enhanced sessions**

## What This Notebook Does

**Phase 0** (cells below) creates all Identity Center resources from scratch:

| Cell | What | Where |
|:-----|:-----|:------|
| P0-1 | Discover IC instance ARN + gateway URL | read-only |
| P0-2 | Create Trusted Token Issuer pointing at Teleport cluster | IC |
| P0-3 | Create Customer Managed Application with `jwt-bearer` grant | IC |
| P0-4 | Create IC users from `DEMO_USER_EMAILS`, assign to Application | IC |
| P0-5 | Write `TTI_APPLICATION_ARN` + `TTI_INSTANCE_ARN` to `.env` | local |

**Steps 1–7** wire up the IAM side and deploy the Lambda updates:

| Step | What | Where |
|:-----|:-----|:------|
| 2 | ~~Register IC as IAM OIDC provider~~ — skipped (IC signing keys are internal) | — |
| 3 | Create `agentcore-user-role` trusted by the interceptor Lambda role | IAM |
| 4 | Add `sso-oauth:CreateTokenWithIAM` + `sts:AssumeRole` to interceptor role | IAM |
| 5 | Add interceptor role ARN to Application Actor Policy | IC |
| 6 | Deploy updated interceptor + tool Lambda with env vars | Lambda |
| 7 | Test — verify `aws_caller.Arn` contains user email; check CloudTrail | — |

## Phase 0: Identity Center Setup

Run these cells if starting from scratch — they create the Trusted Token Issuer and
Customer Managed Application that the rest of the notebook requires.

**Skip to Step 1** if you already ran Phase 0 on a previous session and your `.env`
already has `TTI_APPLICATION_ARN` and `TTI_INSTANCE_ARN` filled in.

> **One manual prerequisite:** Identity-Enhanced Sessions must be enabled in the
> IAM Identity Center console before `sso-oidc:CreateTokenWithIAM` will work.
> Console path: **IAM Identity Center → Settings → Enable identity-enhanced sessions**

In [ ]:
# Phase 0 — minimal bootstrap (separate from the main setup cell so it can run first)
import os, re
import boto3
from dotenv import load_dotenv

load_dotenv(dotenv_path='.env', override=True)
os.environ.setdefault('AWS_DEFAULT_REGION', 'us-east-1')

_s = boto3.Session(
    aws_access_key_id=os.environ.get('AWS_ACCESS_KEY_ID'),
    aws_secret_access_key=os.environ.get('AWS_SECRET_ACCESS_KEY'),
    aws_session_token=os.environ.get('AWS_SESSION_TOKEN'),
    region_name=os.environ['AWS_DEFAULT_REGION'],
)

TELEPORT_CLUSTER = os.environ['TELEPORT_CLUSTER']
RESOURCE_PREFIX  = os.environ.get('RESOURCE_PREFIX', 'teleport-demo')
IC_REGION        = os.environ.get('IDENTITY_CENTER_REGION', 'us-east-1')
GATEWAY_NAME     = f'{RESOURCE_PREFIX}-identity-demo'
TTI_NAME         = f'teleport-{TELEPORT_CLUSTER.split(".")[0]}'
APP_NAME_IC      = f'{RESOURCE_PREFIX}-tti-app'

_sso  = _s.client('sso-admin', region_name=IC_REGION)
_gw   = _s.client('bedrock-agentcore-control')

# Discover the Identity Center instance (exactly one per account)
instances = _sso.list_instances()['Instances']
assert instances, 'No Identity Center instance found — ensure IC is enabled in this account'
IC_INSTANCE_ARN   = instances[0]['InstanceArn']
IC_IDENTITY_STORE = instances[0]['IdentityStoreId']

# Fetch gateway URL from AgentCore API (e.g. https://<id>.gateway.bedrock-agentcore.../mcp)
gateways = _gw.list_gateways(maxResults=100)['items']
gw = next((g for g in gateways if g['name'] == GATEWAY_NAME), None)
assert gw, f'Gateway {GATEWAY_NAME!r} not found — complete notebooks 01-03 first'
GATEWAY_URL = _gw.get_gateway(gatewayIdentifier=gw['gatewayId'])['gatewayUrl']

# Teleport prepends "mcp+" to the app URI in teleport.yaml, so the JWT aud claim is
# "mcp+https://..." — AuthorizedAudiences must match this exactly.
# Normalize to always include /mcp regardless of what the API returns
_base = GATEWAY_URL.rstrip('/').removesuffix('/mcp')
MCP_AUDIENCE = f'mcp+{_base}/mcp'

print(f'IC instance ARN  : {IC_INSTANCE_ARN}')
print(f'Identity store   : {IC_IDENTITY_STORE}')
print(f'TTI issuer URL   : https://{TELEPORT_CLUSTER}')
print(f'Gateway URL      : {GATEWAY_URL}')
print(f'JWT audience     : {MCP_AUDIENCE}')
print()
print('Confirm identity-enhanced sessions are ON before continuing:')
print('   IAM Identity Center -> Settings -> Enable identity-enhanced sessions')


In [ ]:
# Phase 0 — Create Trusted Token Issuer (idempotent: skips if already exists by name)
from botocore.exceptions import ClientError

existing_ttis = _sso.list_trusted_token_issuers(InstanceArn=IC_INSTANCE_ARN).get('TrustedTokenIssuers', [])
existing_tti  = next((t for t in existing_ttis if t['Name'] == TTI_NAME), None)

if existing_tti:
    TTI_ARN = existing_tti['TrustedTokenIssuerArn']
    print(f'✓ TTI already exists: {TTI_ARN}')
else:
    resp = _sso.create_trusted_token_issuer(
        InstanceArn=IC_INSTANCE_ARN,
        Name=TTI_NAME,
        TrustedTokenIssuerType='OIDC_JWT',
        TrustedTokenIssuerConfiguration={
            'OidcJwtConfiguration': {
                'IssuerUrl':                    f'https://{TELEPORT_CLUSTER}',
                'ClaimAttributePath':            'sub',
                'IdentityStoreAttributePath':    'emails.value',
                'JwksRetrievalOption':           'OPEN_ID_DISCOVERY',
            }
        }
    )
    TTI_ARN = resp['TrustedTokenIssuerArn']
    print(f'✓ Created TTI: {TTI_ARN}')

print(f'  IssuerUrl : https://{TELEPORT_CLUSTER}')
print(f'  sub → emails.value (Teleport email maps to IC user email)')

In [ ]:
# Phase 0 — Create Customer Managed Application + jwt-bearer grant (idempotent)
_account_id   = _s.client('sts').get_caller_identity()['Account']
_account_root = f'arn:aws:iam::{_account_id}:root'

existing_apps = _sso.list_applications(InstanceArn=IC_INSTANCE_ARN).get('Applications', [])
existing_app  = next((a for a in existing_apps if a['Name'] == APP_NAME_IC), None)

if existing_app:
    TTI_APPLICATION_ARN = existing_app['ApplicationArn']
    print(f'Application already exists: {TTI_APPLICATION_ARN}')
else:
    resp = _sso.create_application(
        InstanceArn=IC_INSTANCE_ARN,
        ApplicationProviderArn='arn:aws:sso::aws:applicationProvider/custom',
        Name=APP_NAME_IC,
        PortalOptions={'Visibility': 'DISABLED'},
    )
    TTI_APPLICATION_ARN = resp['ApplicationArn']
    print(f'Created Application: {TTI_APPLICATION_ARN}')

    # Set initial actor policy — account root allows manual testing;
    # the interceptor Lambda role is merged in later by Step 5
    _sso.put_application_authentication_method(
        ApplicationArn=TTI_APPLICATION_ARN,
        AuthenticationMethodType='IAM',
        AuthenticationMethod={
            'Iam': {
                'ActorPolicy': {
                    'Version': '2012-10-17',
                    'Statement': [{
                        'Effect': 'Allow',
                        'Principal': {'AWS': _account_root},
                        'Action': 'sso-oauth:CreateTokenWithIAM',
                        'Resource': TTI_APPLICATION_ARN,
                    }]
                }
            }
        }
    )
    print(f'  Actor policy: {_account_root}')

# Always refresh the jwt-bearer grant with the current MCP_AUDIENCE
# MCP_AUDIENCE = "mcp+<gateway_url>" — must match the aud claim in the Teleport JWT exactly
_sso.put_application_grant(
    ApplicationArn=TTI_APPLICATION_ARN,
    GrantType='urn:ietf:params:oauth:grant-type:jwt-bearer',
    Grant={
        'JwtBearer': {
            'AuthorizedTokenIssuers': [{
                'TrustedTokenIssuerArn': TTI_ARN,
                'AuthorizedAudiences':   [MCP_AUDIENCE],
            }]
        }
    }
)
print(f'jwt-bearer grant set (audience: {MCP_AUDIENCE})')
print(f'\nTTI_APPLICATION_ARN = {TTI_APPLICATION_ARN}')


In [ ]:
# Phase 0 — Assign demo users to the Application
# Set DEMO_USER_EMAILS in .env (comma-separated) — typically the emails of everyone who
# will run the demo. Each email must match the user's Teleport `sub` claim (their email).
# Users are created in IC if they don't exist; skip any that can't be resolved.

import boto3 as _b3

DEMO_USER_EMAILS = [
    e.strip() for e in os.environ.get('DEMO_USER_EMAILS', '').split(',') if e.strip()
]

if not DEMO_USER_EMAILS:
    print('ℹ  DEMO_USER_EMAILS not set in .env — skipping user assignment')
    print('   Set DEMO_USER_EMAILS=user1@example.com,user2@example.com and re-run')
else:
    _id_store = _s.client('identitystore', region_name=IC_REGION)

    for email in DEMO_USER_EMAILS:
        # Look up user by email in the Identity Store
        results = _id_store.list_users(
            IdentityStoreId=IC_IDENTITY_STORE,
            Filters=[{'AttributePath': 'UserName', 'AttributeValue': email}]
        )['Users']

        if results:
            user_id = results[0]['UserId']
            print(f'  found  {email} → {user_id}')
        else:
            # Create the user so TTI exchange can map the Teleport sub to an IC identity
            given, family = (email.split('@')[0].split('.')[:2] + [''])[:2], ''
            resp = _id_store.create_user(
                IdentityStoreId=IC_IDENTITY_STORE,
                UserName=email,
                DisplayName=email,
                Name={'GivenName': given[0].capitalize(), 'FamilyName': given[1].capitalize() if len(given) > 1 else ''},
                Emails=[{'Value': email, 'Type': 'work', 'Primary': True}],
            )
            user_id = resp['UserId']
            print(f'  created {email} → {user_id}')

        # Assign user to Application (idempotent — errors on duplicate assignment)
        try:
            _sso.create_application_assignment(
                ApplicationArn=TTI_APPLICATION_ARN,
                PrincipalId=user_id,
                PrincipalType='USER',
            )
            print(f'  ✓ assigned to application')
        except _sso.exceptions.ConflictException:
            print(f'  ✓ already assigned')
        except ClientError as e:
            print(f'  ✗ assignment failed: {e.response["Error"]["Code"]}')

In [ ]:
# Phase 0 — Write discovered/created ARNs to .env and reload
# After this cell the main setup cell (Step 1) can read TTI_APPLICATION_ARN and
# TTI_INSTANCE_ARN from .env without any manual editing.

def _update_env_file(path, updates):
    try:
        with open(path) as f:
            content = f.read()
    except FileNotFoundError:
        content = ''
    for key, value in updates.items():
        pattern = rf'^{re.escape(key)}=.*$'
        line    = f'{key}={value}'
        if re.search(pattern, content, re.MULTILINE):
            content = re.sub(pattern, line, content, flags=re.MULTILINE)
        else:
            content = content.rstrip('\n') + f'\n{line}\n'
    with open(path, 'w') as f:
        f.write(content)

_update_env_file('.env', {
    'TTI_APPLICATION_ARN':    TTI_APPLICATION_ARN,
    'TTI_INSTANCE_ARN':       IC_INSTANCE_ARN,
    'IDENTITY_CENTER_REGION': IC_REGION,
})

# Reload so the Step 1 setup cell picks up the values immediately
load_dotenv(dotenv_path='.env', override=True)

print('✓ .env updated')
print(f'  TTI_APPLICATION_ARN    = {TTI_APPLICATION_ARN}')
print(f'  TTI_INSTANCE_ARN       = {IC_INSTANCE_ARN}')
print(f'  IDENTITY_CENTER_REGION = {IC_REGION}')
print()
print('→ Continue to Step 1')

## Step 1: Setup

In [ ]:
import os, json, time, io, zipfile, subprocess
import boto3
from dotenv import load_dotenv
from botocore.exceptions import ClientError

load_dotenv(dotenv_path='.env', override=True)
os.environ.setdefault('AWS_DEFAULT_REGION', 'us-east-1')
REGION = os.environ['AWS_DEFAULT_REGION']

session = boto3.Session(
    aws_access_key_id=os.environ.get('AWS_ACCESS_KEY_ID'),
    aws_secret_access_key=os.environ.get('AWS_SECRET_ACCESS_KEY'),
    aws_session_token=os.environ.get('AWS_SESSION_TOKEN'),
    region_name=REGION
)

RESOURCE_PREFIX          = os.environ.get('RESOURCE_PREFIX', 'teleport-demo')
TTI_APPLICATION_ARN      = os.environ['TTI_APPLICATION_ARN']
TTI_INSTANCE_ARN         = os.environ['TTI_INSTANCE_ARN']
IDENTITY_CENTER_REGION   = os.environ.get('IDENTITY_CENTER_REGION', 'us-east-1')
TAG_CREATOR              = os.environ.get('TAG_CREATOR', '')
TAG_DEMO                 = os.environ.get('TAG_DEMO', 'teleport-agentcore')
TARGET_NAME              = 'TeleportDemo'
APP_NAME                 = 'agentcore-gateway'  # must match app name in teleport.yaml

# Derive Identity Center OIDC issuer URL from the instance ARN
# Instance ARN format: arn:aws:sso:::instance/ssoins-<id>
INSTANCE_ID     = TTI_INSTANCE_ARN.split('/')[-1]
IC_ISSUER_URL   = f'https://identitycenter.amazonaws.com/{INSTANCE_ID}'

iam            = session.client('iam')
lambda_client  = session.client('lambda')
gateway_client = session.client('bedrock-agentcore-control')
sso_admin      = session.client('sso-admin', region_name=IDENTITY_CENTER_REGION)
account_id     = session.client('sts').get_caller_identity()['Account']

# Derived resource names (same prefix convention as notebooks 01-03)
INTERCEPTOR_ROLE_NAME   = f'{RESOURCE_PREFIX}-interceptor-lambda-role'
INTERCEPTOR_LAMBDA_NAME = f'{RESOURCE_PREFIX}-interceptor'
TOOL_LAMBDA_NAME        = f'{RESOURCE_PREFIX}-tool-lambda'
USER_ROLE_NAME          = f'{RESOURCE_PREFIX}-agentcore-user-role'
GATEWAY_NAME            = f'{RESOURCE_PREFIX}-identity-demo'

# Resolve existing ARNs and print current state
interceptor_role_arn = iam.get_role(RoleName=INTERCEPTOR_ROLE_NAME)['Role']['Arn']
interceptor_lambda_arn = lambda_client.get_function(
    FunctionName=INTERCEPTOR_LAMBDA_NAME)['Configuration']['FunctionArn']
tool_lambda_arn = lambda_client.get_function(
    FunctionName=TOOL_LAMBDA_NAME)['Configuration']['FunctionArn']

gateways = gateway_client.list_gateways(maxResults=100)['items']
gw = next((g for g in gateways if g['name'] == GATEWAY_NAME), None)
gateway_id  = gw['gatewayId'] if gw else None
gateway_url = gateway_client.get_gateway(
    gatewayIdentifier=gateway_id)['gatewayUrl'] if gateway_id else None

print(f'Account:              {account_id}')
print(f'Region:               {REGION}')
print(f'IC Region:            {IDENTITY_CENTER_REGION}')
print(f'IC Issuer URL:        {IC_ISSUER_URL}')
print(f'TTI Application ARN:  {TTI_APPLICATION_ARN}')
print()
print(f'Interceptor Role:     {interceptor_role_arn}')
print(f'Interceptor Lambda:   {interceptor_lambda_arn}')
print(f'Tool Lambda:          {tool_lambda_arn}')
print(f'Gateway ID:           {gateway_id}')

## Step 2: IAM OIDC Provider — Skipped

IC tokens from `CreateTokenWithIAM` are signed with **internal keys**. The IC OIDC
discovery endpoint (`/.well-known/openid-configuration`) returns 404 — the signing
keys are not publicly accessible — so `sts:AssumeRoleWithWebIdentity` cannot verify
the token signature and this step is not needed.

Instead, Step 3 grants the interceptor Lambda's execution role permission to call
`sts:AssumeRole` directly. The IC TTI exchange still validates the user through
Identity Center before the role assumption runs — it just doesn't federate the
IC token into STS.

In [ ]:
# Step 2 — No-op: OIDC provider not needed for this flow.
# IC tokens from CreateTokenWithIAM use internal signing keys that are not published
# via the standard OIDC discovery endpoint, so AssumeRoleWithWebIdentity cannot
# verify them. We use sts:AssumeRole directly in Step 3/4 instead.
print('Step 2 skipped — OIDC provider not required')
OIDC_PROVIDER_ARN = None  # unused in subsequent steps

## Step 3: Create the User-Scoped IAM Role

This role is assumed by the interceptor Lambda on behalf of the calling user.
The interceptor sets `RoleSessionName` to the user's email, so the ARN in CloudTrail
becomes `assumed-role/<role>/user@example.com` — proof that API calls are attributed
to the user, not the agent.

The trust policy grants the interceptor Lambda execution role permission to call
`sts:AssumeRole`. The IC TTI exchange (step 1 inside the interceptor) validates the
user's identity in Identity Center before the role assumption is attempted.

Permissions are minimal for the demo: just `sts:GetCallerIdentity`.

In [ ]:
# Trust policy: interceptor Lambda execution role can AssumeRole
# RoleSessionName is set to the user's email by the interceptor → CloudTrail attribution
trust_policy = {
    'Version': '2012-10-17',
    'Statement': [{
        'Effect': 'Allow',
        'Principal': {'AWS': interceptor_role_arn},
        'Action': 'sts:AssumeRole',
    }]
}

try:
    resp = iam.create_role(
        RoleName=USER_ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust_policy),
        Description='User-scoped role for Teleport AgentCore TTI identity demo',
    )
    USER_ROLE_ARN = resp['Role']['Arn']
    print(f'✓ Created role: {USER_ROLE_ARN}')
    time.sleep(10)  # IAM eventual consistency
except ClientError as e:
    if e.response['Error']['Code'] == 'EntityAlreadyExists':
        USER_ROLE_ARN = iam.get_role(RoleName=USER_ROLE_NAME)['Role']['Arn']
        # Always refresh trust policy in case it was created with old config
        iam.update_assume_role_policy(
            RoleName=USER_ROLE_NAME,
            PolicyDocument=json.dumps(trust_policy),
        )
        print(f'✓ Role already exists (trust policy refreshed): {USER_ROLE_ARN}')
    else:
        raise

# Minimal permissions for the demo — expand here to let users make real AWS calls
iam.put_role_policy(
    RoleName=USER_ROLE_NAME,
    PolicyName='agentcore-user-demo-permissions',
    PolicyDocument=json.dumps({
        'Version': '2012-10-17',
        'Statement': [{
            'Effect': 'Allow',
            'Action': 'sts:GetCallerIdentity',
            'Resource': '*'
        }]
    })
)
print('✓ Attached demo permissions policy (sts:GetCallerIdentity)')
print(f'\nUSER_ROLE_ARN = {USER_ROLE_ARN}')

## Step 4: Grant the Interceptor Permission to Exchange Tokens

The interceptor Lambda needs two new permissions:
- `sso-oauth:CreateTokenWithIAM` — to call Identity Center and validate the Teleport user
- `sts:AssumeRole` — to assume the user-scoped role with the user's email as `RoleSessionName`

Both are scoped to the specific resources created in this notebook.

In [ ]:
iam.put_role_policy(
    RoleName=INTERCEPTOR_ROLE_NAME,
    PolicyName='tti-token-exchange',
    PolicyDocument=json.dumps({
        'Version': '2012-10-17',
        'Statement': [
            {
                'Effect': 'Allow',
                'Action': 'sso-oauth:CreateTokenWithIAM',
                'Resource': TTI_APPLICATION_ARN
            },
            {
                'Effect': 'Allow',
                'Action': 'sts:AssumeRole',
                'Resource': USER_ROLE_ARN
            }
        ]
    })
)
print(f'✓ Added tti-token-exchange policy to {INTERCEPTOR_ROLE_NAME}')
print(f'  sso-oauth:CreateTokenWithIAM → {TTI_APPLICATION_ARN}')
print(f'  sts:AssumeRole               → {USER_ROLE_ARN}')

## Step 5: Update the Application Actor Policy

The Identity Center Application controls which IAM principals can call
`sso-oauth:CreateTokenWithIAM`. The interceptor Lambda's execution role must be
listed or the token exchange returns `UnauthorizedClient`.

`put_application_authentication_method` replaces the entire policy, so this cell
reads the existing principals and merges the interceptor role in.

In [ ]:
# Read existing actor policy to avoid overwriting other principals
try:
    current = sso_admin.get_application_authentication_method(
        ApplicationArn=TTI_APPLICATION_ARN,
        AuthenticationMethodType='IAM'
    )
    existing_policy = current['AuthenticationMethod']['Iam']['ActorPolicy']
    stmts = existing_policy.get('Statement', [])
    # Collect all existing principals (handle both string and list formats)
    existing_principals = []
    for stmt in stmts:
        p = stmt.get('Principal', {}).get('AWS', [])
        if isinstance(p, str):
            existing_principals.append(p)
        else:
            existing_principals.extend(p)
    print(f'Existing principals: {existing_principals}')
except ClientError:
    existing_principals = []
    print('No existing actor policy found — creating fresh')

# Merge interceptor role; keep account root for manual testing
account_root = f'arn:aws:iam::{account_id}:root'
merged = list({*existing_principals, interceptor_role_arn, account_root})

sso_admin.put_application_authentication_method(
    ApplicationArn=TTI_APPLICATION_ARN,
    AuthenticationMethodType='IAM',
    AuthenticationMethod={
        'Iam': {
            'ActorPolicy': {
                'Version': '2012-10-17',
                'Statement': [{
                    'Effect': 'Allow',
                    'Principal': {'AWS': merged},
                    'Action': 'sso-oauth:CreateTokenWithIAM',
                    'Resource': TTI_APPLICATION_ARN
                }]
            }
        }
    }
)
print(f'\n✓ Application Actor Policy updated')
for p in merged:
    print(f'  {p}')

## Step 6: Deploy Updated Lambda Code

**Before running this step**, ensure both Lambda source files have been updated:
- `lambda_interceptor_avp.py` — add `_exchange_token_for_credentials()` after AVP ALLOW
- `lambda_tool.py` — update `whoami_tool` to call `sts:GetCallerIdentity` with user creds

See `PROPOSAL-tti-lambda-identity.md` for the exact code changes required.

The interceptor cell also merges in the three new environment variables while
preserving the existing `AVP_POLICY_STORE_ID`.

In [ ]:
# Read current env vars so AVP_POLICY_STORE_ID is preserved
current_config = lambda_client.get_function_configuration(
    FunctionName=INTERCEPTOR_LAMBDA_NAME)
current_env = current_config.get('Environment', {}).get('Variables', {})

merged_env = {
    **current_env,
    'TTI_APPLICATION_ARN':    TTI_APPLICATION_ARN,
    'USER_ROLE_ARN':          USER_ROLE_ARN,
    'IDENTITY_CENTER_REGION': IDENTITY_CENTER_REGION,
}

lambda_client.update_function_configuration(
    FunctionName=INTERCEPTOR_LAMBDA_NAME,
    Environment={'Variables': merged_env}
)
lambda_client.get_waiter('function_updated_v2').wait(FunctionName=INTERCEPTOR_LAMBDA_NAME)
print('✓ Environment variables updated')
for k, v in merged_env.items():
    print(f'  {k} = {v}')

# Package and deploy the updated interceptor
with open('lambda_interceptor_avp.py') as f:
    code = f.read()
buf = io.BytesIO()
with zipfile.ZipFile(buf, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.writestr('lambda_function.py', code)
buf.seek(0)

lambda_client.update_function_code(
    FunctionName=INTERCEPTOR_LAMBDA_NAME, ZipFile=buf.read())
lambda_client.get_waiter('function_updated_v2').wait(FunctionName=INTERCEPTOR_LAMBDA_NAME)
print(f'\n✓ Interceptor Lambda deployed ({INTERCEPTOR_LAMBDA_NAME})')

In [ ]:
with open('lambda_tool.py') as f:
    code = f.read()
buf = io.BytesIO()
with zipfile.ZipFile(buf, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.writestr('lambda_function.py', code)
buf.seek(0)

lambda_client.update_function_code(
    FunctionName=TOOL_LAMBDA_NAME, ZipFile=buf.read())
lambda_client.get_waiter('function_updated_v2').wait(FunctionName=TOOL_LAMBDA_NAME)
print(f'✓ Tool Lambda deployed ({TOOL_LAMBDA_NAME})')

In [ ]:
print('Waiting for gateway to reach READY status...')
while True:
    status = gateway_client.get_gateway(gatewayIdentifier=gateway_id)['status']
    print(f'  {status}')
    if status == 'READY':
        break
    if status == 'FAILED':
        raise RuntimeError('Gateway is in FAILED state — check CloudWatch logs')
    time.sleep(10)
print('✓ Gateway READY')

## Step 7: Test — Two Identities Side by Side

`whoami_tool` now returns two ARNs:
- `lambda_execution_role` — the static IAM role the Lambda was deployed with
- `aws_caller` — a temporary assumed-role session named after the Teleport user

The session name in `aws_caller.Arn` contains the user's email. That's the proof:
the tool Lambda's AWS API calls are attributed to the user in CloudTrail.

In [ ]:
import json as _json

def call_tool(tool_name, arguments={}):
    """Call a gateway tool via tsh mcp connect and return the parsed MCP response."""
    msgs = [
        '{"jsonrpc":"2.0","id":1,"method":"initialize","params":{"protocolVersion":"2024-11-05","capabilities":{},"clientInfo":{"name":"notebook","version":"0.0.1"}}}',
        _json.dumps({
            'jsonrpc': '2.0', 'id': 2, 'method': 'tools/call',
            'params': {'name': f'{TARGET_NAME}___{tool_name}', 'arguments': arguments}
        })
    ]
    result = subprocess.run(
        ['tsh', 'mcp', 'connect', APP_NAME],
        input='\n'.join(msgs) + '\n',
        capture_output=True, text=True, timeout=30
    )
    for line in result.stdout.splitlines():
        try:
            msg = _json.loads(line)
            if msg.get('id') == 2:
                return msg
        except Exception:
            pass
    return None

def parse_tool_body(msg):
    """Unwrap the nested MCP response envelope to get the tool's return value."""
    if msg is None:
        return None
    if 'error' in msg:
        return {'_error': msg['error']['message']}
    text = msg['result']['content'][0]['text']
    outer = _json.loads(text)
    return _json.loads(outer['body'])

print('✓ Helpers loaded')

In [ ]:
msg    = call_tool('whoami_tool')
result = parse_tool_body(msg)

print('whoami_tool response:')
print(_json.dumps(result, indent=2))
print()

# Assertions — these should pass when the TTI exchange is working correctly
lambda_role   = result.get('lambda_execution_role', '')
aws_caller    = result.get('aws_caller', {})
aws_arn       = aws_caller.get('Arn', '') if isinstance(aws_caller, dict) else ''
teleport_user = result.get('teleport_caller', '')

# get_caller_identity() returns arn:aws:sts::... for assumed-role sessions (not arn:aws:iam)
assert lambda_role and 'arn:aws:' in lambda_role, \
    'FAIL — lambda_execution_role missing or malformed'
assert 'assumed-role' in aws_arn, \
    f'FAIL — aws_caller.Arn is not an assumed-role session: {aws_arn}\n' \
    f'       (check that Step 6 ran and that TTI_APPLICATION_ARN + USER_ROLE_ARN are set on the interceptor Lambda)'
assert teleport_user and teleport_user in aws_arn, \
    f'FAIL — Teleport user email ({teleport_user}) not found in aws_caller ARN ({aws_arn})'
assert lambda_role != aws_arn, \
    'FAIL — lambda_execution_role and aws_caller are the same identity'

print(f'✓ PASS  lambda_execution_role : {lambda_role}')
print(f'✓ PASS  aws_caller.Arn        : {aws_arn}')
print(f'✓ PASS  user email in ARN     : {teleport_user}')

In [ ]:
# CloudTrail — confirm the GetCallerIdentity call appears under the user's session
ct = session.client('cloudtrail')
events = ct.lookup_events(
    LookupAttributes=[{'AttributeKey': 'EventName', 'AttributeValue': 'GetCallerIdentity'}],
    MaxResults=5,
)
print('Recent GetCallerIdentity events:')
for e in events['Events']:
    detail = _json.loads(e['CloudTrailEvent'])
    arn    = detail.get('userIdentity', {}).get('arn', 'unknown')
    print(f'  {e["EventTime"]}  {arn}')

print()
print('Look for: assumed-role/' + USER_ROLE_NAME + '/<user-email>')

## What Just Happened

| Layer | Identity |
|:------|:---------|
| Teleport JWT `sub` | `jeffrey.ellin@goteleport.com` |
| Identity Center token `sub` | `<identity-center-user-id>` |
| STS session name | `jeffrey.ellin@goteleport.com` (set by interceptor via `RoleSessionName`) |
| `aws_caller.Arn` | `arn:aws:sts::<account>:assumed-role/<role>/jeffrey.ellin@goteleport.com` |
| CloudTrail actor | `assumed-role/<role>/jeffrey.ellin@goteleport.com` |

**The chain is unbroken.** The Teleport user identity flows through every layer:

1. Teleport validates the user and issues a signed JWT (`sub` = user email)
2. AVP checks that the user's Teleport role is allowed to call the tool (Cedar ALLOW)
3. The interceptor exchanges the Teleport JWT for an Identity Center token via TTI
4. The interceptor calls STS to assume a role, naming the session after the user
5. The tool Lambda uses those credentials — CloudTrail attributes every API call to the user

No ambient permissions. No service account standing in for the user. Zero-trust, end to end.

## Cleanup

Removes all resources created by this notebook — including the Identity Center
Application and Trusted Token Issuer created in Phase 0. Run this to return the
account to the state it was in after completing notebooks 01–03.

In [ ]:
# ── Cleanup: safe to run at any point, even in a fresh kernel ──────────────
import os, io, zipfile, json as _j, re
import boto3
from dotenv import load_dotenv
from botocore.exceptions import ClientError

load_dotenv(dotenv_path='.env', override=True)
os.environ.setdefault('AWS_DEFAULT_REGION', 'us-east-1')
_s = boto3.Session(
    aws_access_key_id=os.environ.get('AWS_ACCESS_KEY_ID'),
    aws_secret_access_key=os.environ.get('AWS_SECRET_ACCESS_KEY'),
    aws_session_token=os.environ.get('AWS_SESSION_TOKEN'),
    region_name=os.environ['AWS_DEFAULT_REGION'],
)
_iam    = _s.client('iam')
_lambda = _s.client('lambda')

_PREFIX           = os.environ.get('RESOURCE_PREFIX', 'teleport-demo')
_IC_REGION        = os.environ.get('IDENTITY_CENTER_REGION', 'us-east-1')
_TTI_APP_ARN      = os.environ.get('TTI_APPLICATION_ARN', '')
_TTI_INSTANCE_ARN = os.environ.get('TTI_INSTANCE_ARN', '')
_INSTANCE_ID      = _TTI_INSTANCE_ARN.split('/')[-1] if _TTI_INSTANCE_ARN else ''
_TELEPORT_CLUSTER = os.environ.get('TELEPORT_CLUSTER', '')
_USER_ROLE        = f'{_PREFIX}-agentcore-user-role'
_INTERCEPTOR_ROLE = f'{_PREFIX}-interceptor-lambda-role'
_INTERCEPTOR_FUNC = f'{_PREFIX}-interceptor'
_TOOL_FUNC        = f'{_PREFIX}-tool-lambda'
_APP_NAME_IC      = f'{_PREFIX}-tti-app'
_TTI_NAME         = f'teleport-{_TELEPORT_CLUSTER.split(".")[0]}' if _TELEPORT_CLUSTER else ''

def _swallow(fn, *args, **kwargs):
    try:
        return fn(*args, **kwargs)
    except ClientError as e:
        print(f'  skip ({e.response["Error"]["Code"]})')
    except Exception as e:
        print(f'  skip ({e})')

# 1. Delete user-scoped IAM role
print('Deleting user-scoped IAM role...')
_swallow(_iam.delete_role_policy, RoleName=_USER_ROLE, PolicyName='agentcore-user-demo-permissions')
_swallow(_iam.delete_role, RoleName=_USER_ROLE)
print(f'  {_USER_ROLE}')

# 2. Remove tti-token-exchange policy from interceptor role
print('Removing TTI policy from interceptor role...')
_swallow(_iam.delete_role_policy, RoleName=_INTERCEPTOR_ROLE, PolicyName='tti-token-exchange')
print(f'  tti-token-exchange removed from {_INTERCEPTOR_ROLE}')

# 3. Delete IAM OIDC provider for Identity Center (not created by this flow, but may
#    exist from earlier experimentation — delete if found, skip silently if not)
print('Removing Identity Center IAM OIDC provider (if it exists)...')
if _INSTANCE_ID:
    _found_oidc = False
    for p in _iam.list_open_id_connect_providers()['OpenIDConnectProviderList']:
        if _INSTANCE_ID in p['Arn']:
            _swallow(_iam.delete_open_id_connect_provider, OpenIDConnectProviderArn=p['Arn'])
            print(f'  deleted {p["Arn"]}')
            _found_oidc = True
    if not _found_oidc:
        print('  not found — skipping')
else:
    print('  TTI_INSTANCE_ARN not set — skipping')

# 4. Revert interceptor env vars (remove TTI vars, preserve AVP_POLICY_STORE_ID)
print('Reverting interceptor environment variables...')
try:
    cfg = _lambda.get_function_configuration(FunctionName=_INTERCEPTOR_FUNC)
    env = cfg.get('Environment', {}).get('Variables', {})
    for k in ('TTI_APPLICATION_ARN', 'USER_ROLE_ARN', 'IDENTITY_CENTER_REGION'):
        env.pop(k, None)
    _lambda.update_function_configuration(FunctionName=_INTERCEPTOR_FUNC, Environment={'Variables': env})
    _lambda.get_waiter('function_updated_v2').wait(FunctionName=_INTERCEPTOR_FUNC)
    print(f'  TTI vars removed from {_INTERCEPTOR_FUNC}')
except ClientError as e:
    print(f'  skip ({e.response["Error"]["Code"]})')

# 5. Redeploy interceptor (TTI exchange is disabled by the removed env vars above)
print('Redeploying interceptor...')
try:
    with open('lambda_interceptor_avp.py') as f:
        code = f.read()
    buf = io.BytesIO()
    with zipfile.ZipFile(buf, 'w', zipfile.ZIP_DEFLATED) as zf:
        zf.writestr('lambda_function.py', code)
    buf.seek(0)
    _lambda.update_function_code(FunctionName=_INTERCEPTOR_FUNC, ZipFile=buf.read())
    _lambda.get_waiter('function_updated_v2').wait(FunctionName=_INTERCEPTOR_FUNC)
    print(f'  {_INTERCEPTOR_FUNC} redeployed')
except Exception as e:
    print(f'  skip ({e})')

# 6. Redeploy tool Lambda
print('Redeploying tool Lambda...')
try:
    with open('lambda_tool.py') as f:
        code = f.read()
    buf = io.BytesIO()
    with zipfile.ZipFile(buf, 'w', zipfile.ZIP_DEFLATED) as zf:
        zf.writestr('lambda_function.py', code)
    buf.seek(0)
    _lambda.update_function_code(FunctionName=_TOOL_FUNC, ZipFile=buf.read())
    _lambda.get_waiter('function_updated_v2').wait(FunctionName=_TOOL_FUNC)
    print(f'  {_TOOL_FUNC} redeployed')
except Exception as e:
    print(f'  skip ({e})')

# 7. Delete IC Application (created in Phase 0)
print('Deleting Identity Center Application...')
if _TTI_APP_ARN:
    _sso = _s.client('sso-admin', region_name=_IC_REGION)
    _swallow(_sso.delete_application, ApplicationArn=_TTI_APP_ARN)
    print(f'  {_TTI_APP_ARN}')
else:
    print('  TTI_APPLICATION_ARN not set — skipping')

# 8. Delete Trusted Token Issuer (created in Phase 0)
print('Deleting Trusted Token Issuer...')
if _TTI_INSTANCE_ARN and _TTI_NAME:
    _sso = _s.client('sso-admin', region_name=_IC_REGION)
    ttis = _sso.list_trusted_token_issuers(InstanceArn=_TTI_INSTANCE_ARN).get('TrustedTokenIssuers', [])
    match = next((t for t in ttis if t['Name'] == _TTI_NAME), None)
    if match:
        _swallow(_sso.delete_trusted_token_issuer, TrustedTokenIssuerArn=match['TrustedTokenIssuerArn'])
        print(f'  {match["TrustedTokenIssuerArn"]}')
    else:
        print(f'  TTI {_TTI_NAME!r} not found — already deleted or never created')
else:
    print('  TTI_INSTANCE_ARN or TELEPORT_CLUSTER not set — skipping')

# 9. Clear TTI values from .env so next run starts fresh
print('Clearing TTI values from .env...')
try:
    with open('.env') as f:
        content = f.read()
    for key in ('TTI_APPLICATION_ARN', 'TTI_INSTANCE_ARN'):
        content = re.sub(rf'^{key}=.*$', f'{key}=', content, flags=re.MULTILINE)
    with open('.env', 'w') as f:
        f.write(content)
    print('  TTI_APPLICATION_ARN and TTI_INSTANCE_ARN cleared')
except Exception as e:
    print(f'  skip ({e})')

print('\nCleanup complete')

### Optional: Delete Identity Center Users

Only run this if Phase 0 created the IC users (i.e., they didn't already exist before
this demo). If your account uses SCIM sync or the users have other IC assignments,
**skip this cell** — the Application deletion above already removed their assignments.

In [ ]:
# Optional cleanup — delete IC users created by Phase 0
# Reads DEMO_USER_EMAILS from .env; silently skips users that don't exist
load_dotenv(dotenv_path='.env', override=True)
_IC_REGION2       = os.environ.get('IDENTITY_CENTER_REGION', 'us-east-1')
_TTI_INSTANCE_ARN2 = os.environ.get('TTI_INSTANCE_ARN', '')

_emails = [e.strip() for e in os.environ.get('DEMO_USER_EMAILS', '').split(',') if e.strip()]
if not _emails:
    print('DEMO_USER_EMAILS not set — nothing to delete')
elif not _TTI_INSTANCE_ARN2:
    print('TTI_INSTANCE_ARN not set — cannot determine Identity Store ID')
else:
    _instances2  = _s.client('sso-admin', region_name=_IC_REGION2).list_instances()['Instances']
    _id_store_id = _instances2[0]['IdentityStoreId']
    _id_store2   = _s.client('identitystore', region_name=_IC_REGION2)

    for email in _emails:
        results = _id_store2.list_users(
            IdentityStoreId=_id_store_id,
            Filters=[{'AttributePath': 'UserName', 'AttributeValue': email}]
        )['Users']
        if results:
            uid = results[0]['UserId']
            try:
                _id_store2.delete_user(IdentityStoreId=_id_store_id, UserId=uid)
                print(f'✓ deleted {email} ({uid})')
            except ClientError as e:
                print(f'✗ {email}: {e.response["Error"]["Code"]}')
        else:
            print(f'  {email} not found — already deleted or never created')